# 04 模型训练与融合对比
**论文章节对应**：第4章 预测模型构建、第5章 实验结果与分析

本notebook依次训练以下5个模型并对比性能：
- **M0**: Ridge 线性基线（对比参考）
- **M1**: 随机森林（RF，GridSearchCV调参）
- **M2**: XGBoost（两步超参数搜索）
- **M3a**: Optimized Blending（OOF权重优化，防数据泄露）
- **M3b**: Stacking（RF+XGB基学习器，Ridge元学习器）

**关键防泄露措施**：
- 预处理通过Pipeline封装，只在训练集fit
- Blending权重基于OOF预测求解
- CV使用Expanding Window，尊重时间顺序

**输入**：`data/processed/train.csv`, `data/processed/test.csv`
**输出**：模型文件 + 图5-1/5-2/5-3/5-7 + 表5-1

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import joblib
import warnings
warnings.filterwarnings('ignore')

from src.preprocessing import year_expanding_cv, build_preprocessor
from src.models import (
    train_ridge, train_random_forest, train_xgboost,
    train_blending, predict_blending, train_stacking
)
from src.evaluation import (
    evaluate_model, bootstrap_mape,
    build_results_table, plot_model_comparison,
    plot_pred_vs_actual, plot_residuals, plot_learning_curve
)

matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

# ── 读取数据与配置 ────────────────────────────────
train_df = pd.read_csv('data/processed/train.csv', encoding='utf-8-sig')
test_df  = pd.read_csv('data/processed/test.csv',  encoding='utf-8-sig')
config   = joblib.load('outputs/saved_models/feature_config.pkl')

all_features = config['all_features']
TARGET       = config['target']

X_train = train_df[all_features]
y_train = train_df[TARGET]
X_test  = test_df[all_features]
y_test  = test_df[TARGET]

print(f'训练集: {X_train.shape}, 测试集: {X_test.shape}')
print(f'目标变量: {TARGET}')

In [ ]:
# Expanding Window CV分割（训练集内部）
print('Expanding Window CV分割：')
cv_splits = list(year_expanding_cv(train_df, year_col='completion_year'))
print(f'总共 {len(cv_splits)} 个Fold')

# 构建预处理器
preprocessor = build_preprocessor(
    numeric_features=config['num_features'],
    categorical_features=config['cat_features'],
    ordinal_features=config['ord_features'],
    binary_features=config['bin_features'],
)

## 4.1 M0: Ridge 线性基线

In [ ]:
print('=' * 50)
print('M0: Ridge 线性基线')
print('=' * 50)

best_ridge = train_ridge(X_train, y_train, cv_splits, preprocessor=build_preprocessor(
    numeric_features=config['num_features'],
    categorical_features=config['cat_features'],
    ordinal_features=config['ord_features'],
    binary_features=config['bin_features'],
))

pred_ridge = best_ridge.predict(X_test)
res_ridge  = evaluate_model(y_test, pred_ridge, 'M0-Ridge')

## 4.2 M1: 随机森林

In [ ]:
print('=' * 50)
print('M1: 随机森林（GridSearchCV）')
print('=' * 50)

best_rf = train_random_forest(X_train, y_train, cv_splits, preprocessor=build_preprocessor(
    numeric_features=config['num_features'],
    categorical_features=config['cat_features'],
    ordinal_features=config['ord_features'],
    binary_features=config['bin_features'],
))

pred_rf = best_rf.predict(X_test)
res_rf  = evaluate_model(y_test, pred_rf, 'M1-随机森林')

## 4.3 M2: XGBoost（两步搜索）

In [ ]:
print('=' * 50)
print('M2: XGBoost（两步超参数搜索）')
print('=' * 50)

best_xgb = train_xgboost(X_train, y_train, cv_splits, preprocessor=build_preprocessor(
    numeric_features=config['num_features'],
    categorical_features=config['cat_features'],
    ordinal_features=config['ord_features'],
    binary_features=config['bin_features'],
))

pred_xgb = best_xgb.predict(X_test)
res_xgb  = evaluate_model(y_test, pred_xgb, 'M2-XGBoost')

## 4.4 M3a: Optimized Blending

In [ ]:
print('=' * 50)
print('M3a: Optimized Blending（OOF权重优化）')
print('=' * 50)
print('注：权重基于OOF预测在训练集内部求解，不接触测试集')

w_rf, w_xgb = train_blending(best_rf, best_xgb, X_train, y_train, cv_splits)

pred_blend = predict_blending(best_rf, best_xgb, w_rf, w_xgb, X_test)
res_blend  = evaluate_model(y_test, pred_blend, 'M3a-Blending')

## 4.5 M3b: Stacking

In [ ]:
print('=' * 50)
print('M3b: Stacking（RF+XGB → Ridge元学习器）')
print('=' * 50)

# 预处理（只在训练集fit）
preprocessor_for_stack = joblib.load('outputs/saved_models/preprocessor.pkl')
X_train_proc = preprocessor_for_stack.transform(X_train)
X_test_proc  = preprocessor_for_stack.transform(X_test)

# 获取最优RF和XGB的裸参数（去掉Pipeline包装）
rf_params  = best_rf.named_steps['regressor'].get_params()
xgb_params = best_xgb.named_steps['regressor'].get_params()

# 将参数加上'regressor__'前缀（train_stacking内部会剥离）
rf_params_prefixed  = {f'regressor__{k}': v for k, v in rf_params.items()}
xgb_params_prefixed = {f'regressor__{k}': v for k, v in xgb_params.items()}

stacking = train_stacking(
    X_train_proc, y_train, cv_splits,
    rf_params=rf_params_prefixed,
    xgb_params=xgb_params_prefixed,
)

pred_stacking = stacking.predict(X_test_proc)
res_stacking  = evaluate_model(y_test, pred_stacking, 'M3b-Stacking')

## 5.1 性能汇总（表5-1）

In [ ]:
print('\n=== 所有模型测试集性能汇总 ===')

results_list = [res_ridge, res_rf, res_xgb, res_blend, res_stacking]
preds_dict   = {
    'M0-Ridge':       pred_ridge,
    'M1-随机森林':    pred_rf,
    'M2-XGBoost':     pred_xgb,
    'M3a-Blending':   pred_blend,
    'M3b-Stacking':   pred_stacking,
}

df_table = build_results_table(
    results_list, y_test.values, preds_dict, n_bootstrap=1000
)

# 保存为CSV（可直接导入Word）
df_table.to_csv('outputs/table5_1_results.csv', index=False, encoding='utf-8-sig')
print('\n✓ 表5-1已保存至 outputs/table5_1_results.csv')

## 5.2 论文图表生成

In [ ]:
# 图5-1: 模型性能对比柱状图
plot_model_comparison(
    results_list,
    save_path='outputs/figures/fig5_1_model_comparison.png'
)

In [ ]:
# 确定最优模型（MAPE最小）
best_result = min(results_list, key=lambda x: x['MAPE'])
best_model_name = best_result['model']
best_pred = preds_dict[best_model_name]

print(f'\n最优模型: {best_model_name}')
print(f'  MAPE = {best_result["MAPE"]:.2f}%')
print(f'  R²   = {best_result["R2"]:.4f}')

# 图5-2: 预测值 vs 实际值
plot_pred_vs_actual(
    y_test, best_pred,
    model_name=best_model_name,
    save_path='outputs/figures/fig5_2_pred_vs_actual.png'
)

In [ ]:
# 图5-3: 残差分布图
plot_residuals(
    y_test, best_pred,
    model_name=best_model_name,
    save_path='outputs/figures/fig5_3_residuals.png'
)

In [ ]:
# 图5-7: 学习曲线（以随机森林为例）
# 注：学习曲线用处理后的X（因为RandomForestRegressor不含Pipeline）
from sklearn.ensemble import RandomForestRegressor
rf_bare = best_rf.named_steps['regressor']
X_train_for_lc = preprocessor_for_stack.transform(X_train)

plot_learning_curve(
    rf_bare,
    X_train_for_lc, y_train,
    cv_splits,
    model_name='随机森林',
    save_path='outputs/figures/fig5_7_learning_curve.png'
)

In [ ]:
# 保存最优模型名称（供notebook 05使用）
joblib.dump(
    {'best_model_name': best_model_name,
     'best_pred': best_pred,
     'y_test': y_test.values},
    'outputs/saved_models/best_model_info.pkl'
)

print('\n=== Notebook 04 完成 ===')
print(f'最优模型: {best_model_name}')
print('生成图表: 图5-1 / 图5-2 / 图5-3 / 图5-7')
print('生成表格: 表5-1（含Bootstrap 95%CI）')